In [69]:
import numpy as np
from numpy.linalg import inv
from sklearn.preprocessing import PolynomialFeatures

In [70]:
X = np.array([[50, 10], 
[40,7], [65,12], [70,5], [75,4]])

y = np.array([[3],[7],[6],[1],[9]])
print(X)
print(y)

[[50 10]
 [40  7]
 [65 12]
 [70  5]
 [75  4]]
[[3]
 [7]
 [6]
 [1]
 [9]]


### Number of polynomial terms
Let **p** be the number of raw features (`X.shape[1]`) and **n** be the degree (`order`). The number of polynomial terms, including the intercept and all powers/interactions up to degree n, is:

$$
\boxed{d = \binom{p+n}{n} = \frac{(p+n)!}{p!\,n!}}
$$

For the current **2 features** and **order 2**:

$$
d = \binom{2+2}{2} = \frac{4!}{2!\,2!} = 6.
$$

The terms are:

$$
1,\quad x_1,\quad x_2,\quad x_1^2,\quad x_1x_2,\quad x_2^2.
$$

`PolynomialFeatures` includes the intercept by default. With `include_bias=False`, subtract **1**. After expansion, `X_poly.shape[1]` gives the actual number of terms and coefficients per output/class.


In [71]:
order = 3
poly = PolynomialFeatures(degree=order)
X_poly = poly.fit_transform(X)

In [72]:
# Keep targets as columns; accepts a 1-D target vector too.
y = np.array(y, dtype=float)
if len(y.shape) == 1:
    y = y.reshape(-1, 1)
assert len(y.shape) == 2 and y.shape[0] == X_poly.shape[0], "X and y must have the same number of rows."

# MANUAL IF/ELSE: select the inverse using the polynomial matrix shape.
m, d = X_poly.shape
if m >= d:
    # Tall or square matrix: LEFT inverse (full column rank required).
    assert np.linalg.matrix_rank(X_poly) == d, "Left inverse does not exist: columns are dependent."
    w = inv(X_poly.T @ X_poly) @ X_poly.T @ y
    print("Using LEFT inverse")
else:
    # Wide matrix: RIGHT inverse (full row rank required).
    assert np.linalg.matrix_rank(X_poly) == m, "Right inverse does not exist: rows are dependent."
    w = X_poly.T @ inv(X_poly @ X_poly.T) @ y
    print("Using RIGHT inverse")

# Display every polynomial term with its weight.
raw_names = [f"x{i + 1}" for i in range(X.shape[1])]
terms = poly.get_feature_names_out(raw_names)
print("Polynomial matrix shape:", X_poly.shape)
for output in range(w.shape[1]):
    if w.shape[1] > 1:
        print(f"\nOutput {output + 1}:")
    for i in range(len(terms)):
        print(f"{terms[i]}: {w[i, output]:.6f}")


Using RIGHT inverse
Polynomial matrix shape: (5, 10)
1: 0.001176
x1: 0.020434
x2: 0.002905
x1^2: 0.007757
x1 x2: -0.001958
x2^2: -0.011765
x1^3: 0.000176
x1^2 x2: -0.008223
x1 x2^2: 0.073414
x2^3: -0.198783


In [73]:
# MSE for each y-column. Defaults to training data.
# For test MSE, replace these with raw test features and their true targets.
X_mse = X
y_mse = y

y_mse_pred = poly.transform(X_mse) @ w
y_true = np.array(y_mse)
if y_true.ndim == 1:
    y_true = y_true.reshape(-1, 1)
assert y_true.shape == y_mse_pred.shape, "True targets and predictions must have the same sample rows and target columns."

MSE = np.mean((y_true - y_mse_pred)**2, axis=0)
print()
for column, mse in enumerate(MSE, start=1):
    print(f"MSE for y-column {column}: {mse:.8f}")



MSE for y-column 1: 0.00000000


Prediction; edit the X_test with features you would like to test with

In [74]:
X_test = np.array([[22, 8]])
X_test_poly = poly.transform(X_test)
y_pred = X_test_poly @ w

print(f"Predicted y: {y_pred[0, 0]:.4f}")

Predicted y: -25.2448


In [75]:
X_predict = np.array([[22, 8]])
assert X_predict.ndim == 2 and X_predict.shape[1] == X.shape[1], "Use one row per sample and the same feature columns as X."
X_predict_b = np.concatenate((np.ones((len(X_predict), 1)), X_predict), axis=1)
y_pred = X_predict_b @ w

print("Predicted y (rows = samples; columns = target columns):")
print(y_pred)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 10 is different from 3)